# Récupérer des documents et citer les sources

Baseline documentaire extractive sur les vrais Markdown du cursus. Pas de génération LLM ni API. Objectif : tester récupération, accès, provenance et abstention. Voir atelier_rag_documentaire.md.

## 1. Ingestion locale et provenance

Les cours seuls sont indexés. Les fenêtres conservent les numéros de lignes, ce qui rend chaque extrait vérifiable. Aucun bloc de code du corpus n’est exécuté.

In [1]:
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
root = Path.cwd().resolve()
while not (root/'README.md').exists() and root != root.parent:
    root = root.parent
files = sorted(root.glob('[0-9][0-9]_*/cours*.md'))
assert len(files) >= 9
chunks = []
for path in files:
    lines = path.read_text(encoding='utf-8').splitlines()
    for start in range(0,len(lines),24):
        text = '\n'.join(lines[start:start+36])
        if text.strip():
            chunks.append({'file':str(path.relative_to(root)),'start':start+1,'end':min(start+36,len(lines)),'text':text})
vectorizer = TfidfVectorizer(strip_accents='unicode',ngram_range=(1,2),sublinear_tf=True)
matrix = vectorizer.fit_transform([c['text'] for c in chunks])
print({'documents':len(files),'passages':len(chunks)})
def retrieve(query,allowed=None,k=3):
    candidates = [i for i,c in enumerate(chunks) if allowed is None or c['file'] in allowed]
    if not candidates: return []
    scores = (matrix[candidates]@vectorizer.transform([query]).T).toarray().ravel()
    order = np.argsort(-scores,kind='stable')[:k]
    return [(float(scores[j]),chunks[candidates[j]]) for j in order]

{'documents': 11, 'passages': 218}


## 2. Calibration puis test

Le seuil est fixé sur deux questions répondables et un contrôle négatif de calibration. Le test comporte des questions différentes. La pertinence est annotée au niveau du document, pas du passage.

In [2]:
calibration = [
    ('imputation données manquantes','01_nature_et_preparation_des_donnees/'),
    ('réseaux convolutifs CNN','03_deep_learning/'),
    ('zyxwvutsrqponmlk',None)]
positive_scores = [retrieve(q)[0][0] for q,target in calibration if target]
negative_scores = [retrieve(q)[0][0] for q,target in calibration if target is None]
threshold = (min(positive_scores)+max(negative_scores))/2
test_questions = [('corrélations Pearson Spearman','01_nature_et_preparation_des_donnees/'),
                  ('Q-learning Bellman','06_apprentissage_par_renforcement/'),
                  ('RAG bases de connaissances','04_ia_agentique/'),
                  ('abcdefghijkxyz',None)]
hits,abstentions = [],[]
for question,target in test_questions:
    found = retrieve(question)
    abstain = found[0][0] < threshold
    abstentions.append(abstain)
    if target: hits.append(any(c['file'].startswith(target) for _,c in found))
    print(question, '→', 'ABSTENTION' if abstain else found[0][1]['file'])
print({'seuil calibré':threshold,'Recall@3 documentaire':np.mean(hits)})
assert abstentions[-1]

corrélations Pearson Spearman → 01_nature_et_preparation_des_donnees/cours_nature_et_preparation_donnees.md
Q-learning Bellman → 06_apprentissage_par_renforcement/cours_apprentissage_par_renforcement.md
RAG bases de connaissances → 04_ia_agentique/cours_ia_agentique.md
abcdefghijkxyz → ABSTENTION
{'seuil calibré': 0.03804779255126751, 'Recall@3 documentaire': np.float64(1.0)}


## 3. Réponse extractive et contrôle d’accès

La réponse cite les passages récupérés. La vérification compare chaque passage au fichier source. Le filtre est appliqué avant le classement.

In [3]:
def answer(question,allowed=None):
    found = retrieve(question,allowed)
    if not found or found[0][0] < threshold:
        return 'Abstention : aucun passage suffisamment proche.'
    parts = []
    for score,c in found:
        source = (root/c['file']).read_text(encoding='utf-8').splitlines()
        assert c['text'] == '\n'.join(source[c['start']-1:c['end']])
        citation = f"{c['file']}:L{c['start']}-L{c['end']}"
        parts.append(f"[{citation}] (similarité {score:.3f})\n{c['text'][:450]}")
    return '\n\n'.join(parts)
print(answer('RAG bases de connaissances'))
allowed = {str(p.relative_to(root)) for p in files if p.parent.name.startswith('01_')}
restricted = retrieve('fine-tuning transfert',allowed)
assert all(c['file'] in allowed for _,c in restricted)
assert answer('abcdefghijkxyz').startswith('Abstention')

[04_ia_agentique/cours_ia_agentique.md:L25-L60] (similarité 0.126)

---

## Table des Matières
1. [La Rupture Agentique : Du Modèle Passif au Système Autonome](#1-la-rupture-agentique--du-modèle-passif-au-système-autonome)
2. [L'Anatomie d'un Agent IA Autonome](#2-lanatomie-dun-agent-ia-autonome)
3. [Les Mécanismes de Raisonnement et de Planification](#3-les-mécanismes-de-raisonnement-et-de-planification)
4. [L'Utilisation d'Outils (Tool Use & Function Calling décortiqué)](#4-lutilisation-doutils-tool-use--funct

[04_ia_agentique/cours_ia_agentique.md:L1-L36] (similarité 0.123)
# Module 4 : L'Intelligence Artificielle Agentique

Pratique complémentaire : l'[atelier documentaire](atelier_rag_documentaire.md) met en œuvre récupération dans les cours, citations et abstention. Il distingue cette chaîne extractive hors ligne d'un RAG avec génération par LLM.

> "Les modèles de fondation ne sont pas la destination finale de l'IA, ils en sont le moteur cognitif. L'agentique transforme ce mote

## Exercice

Proposez une paraphrase sans les mots du titre ; ajoutez des questions hors domaine avec des mots ordinaires. Pourquoi le test négatif actuel est-il facile ?

In [4]:
# Écrivez votre expérience ici avant de lire la correction.

## Correction et limites

La chaîne inconnue produit un vecteur nul ; elle ne teste pas les faux positifs lexicaux réalistes. Ajouter des contrôles négatifs proches, des annotations par passage et des questions contradictoires. Le TP mesure une récupération réelle et une réponse extractive, pas la résistance à l’injection ou la fidélité d’un LLM génératif.